# SI7016 — Clase 03 — Lab A (variante Hugging Face)
## Agente con smolagents: tools + memoria

**Objetivo:** construir el mismo agente del Lab A original (tools + inspección de tool calling + memoria conversacional), pero usando el stack nativo de Hugging Face — [`smolagents`](https://huggingface.co/docs/smolagents) — en vez de LangChain/LangGraph.

> **Nota comparativa para la clase:** la diferencia de plataforma no es solo de sintaxis. El `create_agent` de LangChain hace *tool calling* estructurado (el LLM devuelve JSON que indica qué tool llamar). `smolagents.CodeAgent` — el agente insignia de Hugging Face — hace que el LLM **escriba código Python** que llama a las tools directamente y lo ejecuta en un sandbox; smolagents también ofrece `ToolCallingAgent` para el estilo JSON clásico, si se prefiere comparar manzanas con manzanas. Esa distinción (agentes que razonan en código vs. agentes que emiten JSON) es justamente uno de los criterios de plataforma que vimos en la lecture de esta clase.


## 0. Configuración

In [22]:
# secrets en Google Colab con userdata.get()
import os
from google.colab import userdata
if userdata.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets and set in os.environ.")
else:
    print("HF_TOKEN not found in Colab Secrets.")
    print("Please add it via the 🔑 Secrets panel on the left (get one at https://huggingface.co/settings/tokens).")


In [23]:
import os
from dotenv import load_dotenv

# To fix the AssertionError, uncomment the line below and paste your token.
# os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
# os.environ["HF_MODEL"] = "meta-llama/Llama-3.3-70B-Instruct"

load_dotenv()

assert os.getenv("HF_TOKEN"), (
    "Please set HF_TOKEN in your environment variables or .env file "
    "(create one at https://huggingface.co/settings/tokens)."
)
# Cualquier modelo de chat servido por HF Inference Providers funciona aquí.
# Ejemplos vigentes en 2026: "meta-llama/Llama-3.3-70B-Instruct" (usado en la doc oficial de smolagents),
# "Qwen/Qwen2.5-Coder-32B-Instruct" (fuerte para CodeAgent), o modelos open-weight más recientes si tu
# proveedor de Inference los expone (Llama 4 Scout, Qwen3.5, etc.).
MODEL_ID = os.getenv("HF_MODEL") or "meta-llama/Llama-3.3-70B-Instruct"
print(MODEL_ID)


## 1. Tools

In [ ]:
!pip install -U smolagents huggingface_hub

In [24]:

from smolagents import tool

@tool
def buscar_curso(pregunta: str) -> str:
    """
    Consulta informacion local de SI7016.

    Args:
        pregunta: la pregunta del usuario sobre el curso.
    """
    base = {
        "temas": "NLP clasico, embeddings, Transformers, RAG y agentes.",
        "clase 03": "Frameworks, agentes, RAG, MCP/A2A, observabilidad y despliegue.",
        "proyecto": "Proyecto integrador de una solucion NLP aplicada.",
    }
    q = pregunta.lower()
    for k, v in base.items():
        if k in q:
            return v
    return "No encontre evidencia en la base local."


@tool
def contar_palabras(texto: str) -> int:
    """
    Cuenta palabras en un texto.

    Args:
        texto: el texto a analizar.
    """
    return len(texto.split())

## 2. Agente

In [25]:
from smolagents import CodeAgent, InferenceClientModel
import os

# Ensure MODEL_ID is defined (copied from cell 3d0634e6)
# This helps if previous cells were not run or kernel state was reset.
MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"

model = InferenceClientModel(model_id=MODEL_ID)  # usa HF Inference Providers; toma HF_TOKEN del entorno
agent = CodeAgent(
    tools=[buscar_curso, contar_palabras],
    model=model,
    instructions="Eres un asistente de SI7016. Responde en espanol y usa tools cuando aporten evidencia o calculo.",
)
resultado = agent.run(
    "¿Que estudia la clase 03 y cuantas palabras tiene 'RAG conecta recuperacion y generacion'?"
)
print(resultado)

## 3. Inspeccionar la ejecucion

En LangChain/LangGraph inspeccionabamos `r["messages"]` y el atributo `tool_calls` (JSON estructurado). En smolagents, la memoria del agente vive en `agent.memory.steps`: cada `ActionStep` trae el **codigo Python** que el modelo escribio (`code_action` / `model_output`), lo que ese codigo observo al ejecutarse (`observations`), y cualquier error.

In [26]:
from smolagents import ActionStep

for step in agent.memory.steps:
    if isinstance(step, ActionStep):
        print(f"\n--- Paso {step.step_number} ---")
        code_action = getattr(step, "code_action", None) or getattr(step, "model_output", None)
        if code_action:
            print("Codigo / salida del modelo:")
            print(code_action)
        tool_calls = getattr(step, "tool_calls", None)
        if tool_calls:
            print("Tool calls:", tool_calls)
        print("Observaciones:", step.observations)
        if step.error:
            print("Error:", step.error)


## 4. Memoria: continuar una conversacion (equivalente al checkpointing de LangGraph)

LangGraph usa un `checkpointer` + `thread_id` para persistir memoria por hilo de conversacion. `smolagents` no trae un checkpointer persistente incorporado, pero resuelve lo mismo de dos formas mas simples:

1. **Mismo hilo:** llamar `agent.run(..., reset=False)` continua la conversacion sobre la memoria ya acumulada en ese objeto `agent`.
2. **Hilos distintos:** cada instancia de `agent` (o cada lista `agent.memory.steps` guardada aparte) es un hilo independiente — el equivalente a un `thread_id` distinto en LangGraph.

In [27]:
# "thread" grupo-01: seguimos la misma conversacion con reset=False
agent.run("Mi tema de interes en esta conversacion es RAG.")
r2 = agent.run("¿Cual era mi tema de interes?", reset=False)
print("grupo-01:", r2)

# "thread" grupo-02: un agente nuevo = memoria distinta (equivalente a otro thread_id)
agent_grupo02 = CodeAgent(tools=[buscar_curso], model=model)
r3 = agent_grupo02.run("¿Cual era mi tema de interes?", reset=False)
print("grupo-02 (no deberia recordarlo):", r3)


## *retos*
1. Agregue una tool real (por ejemplo, una consulta a la Hugging Face Hub API con `huggingface_hub.list_models`).

2. Compare la memoria de `agent` (grupo-01) y `agent_grupo02`: confirme que son independientes, igual que dos `thread_id` distintos en LangGraph.

3. Fuerce un caso sin tools (una pregunta que el agente pueda responder sin llamarlas) y observe que el "codigo" que genera el paso simplemente devuelve la respuesta.

4. Explique, con sus propias palabras, que aporta el enfoque de **CodeAgent** (el modelo escribe y ejecuta codigo) frente al **tool calling JSON estructurado** de `ToolCallingAgent` (smolagents) o de `create_agent` (LangChain). Pista: piense en expresividad (loops, condicionales, encadenar resultados) vs. seguridad/previsibilidad.

5. Extension: cargue tools desde el servidor MCP del Lab C usando `ToolCollection.from_mcp(...)` (transporte `streamable-http`) y paselas al `CodeAgent` junto con `buscar_curso`/`contar_palabras`.

### Como configurar variables de entorno en Google Colab

Hay dos formas principales de definir variables de entorno en Colab:

1. **Para informacion sensible (tokens): use Colab Secrets.**
   Es el metodo mas seguro porque mantiene las claves fuera del codigo del notebook.
   * Haga clic en el icono "🔑" (Secrets) del panel izquierdo.
   * Haga clic en "Add new secret".
   * Nombre la variable `HF_TOKEN`.
   * Pegue su token (creado en https://huggingface.co/settings/tokens, permiso de lectura basta).
   * Active "Notebook access" para este notebook.
   * Accedalo en Python con `userdata.get('HF_TOKEN')` (ver celda de configuracion arriba).

2. **Para variables no sensibles: use `os.environ` en una celda de codigo.**
   ```python
   import os
   os.environ['MY_VARIABLE'] = 'my_value'
   ```

Si ve el `AssertionError` de la celda de configuracion, use el metodo 1 (Colab Secrets) para definir `HF_TOKEN`.